<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [2]:
# 1. Consolidación y limpieza de datos

# a) Leer y consolidar los archivos anuales
lista_df = []

for archivo in archivos_anio:
    temp = pd.read_csv(archivo)
    temp.columns = temp.columns.str.lower().str.strip()
    lista_df.append(temp)

df_anio = pd.concat(lista_df, ignore_index=True)

# Normalizar nombres de columnas del archivo de códigos
df_codigos.columns = df_codigos.columns.str.lower().str.strip()

# Revisar columnas disponibles
print("Columnas df_anio:")
print(df_anio.columns.tolist())

print("\nColumnas df_codigos:")
print(df_codigos.columns.tolist())

# b) Identificar códigos ISO asociados a más de un país
duplicados_codigo = (
    df_codigos
    .groupby("codigo_iso")["pais"]
    .nunique()
    .reset_index(name="cantidad_paises")
    .query("cantidad_paises > 1")
)

print("\nCódigos ISO asociados a más de un país:")
display(duplicados_codigo)

if not duplicados_codigo.empty:
    codigos_conflictivos = duplicados_codigo["codigo_iso"].tolist()

    print("\nRegistros conflictivos:")
    display(df_codigos[df_codigos["codigo_iso"].isin(codigos_conflictivos)].sort_values("codigo_iso"))

# Eliminar registros inconsistentes conservando una sola relación código-país.
# Criterio: primero se eliminan duplicados exactos y luego se conserva el primer país asociado a cada código ISO.
df_codigos_limpio = (
    df_codigos
    .drop_duplicates()
    .sort_values(["codigo_iso", "pais"])
    .drop_duplicates(subset=["codigo_iso"], keep="first")
)

print("\nCantidad de registros en df_codigos original:", len(df_codigos))
print("Cantidad de registros en df_codigos limpio:", len(df_codigos_limpio))

# c) Combinar df_anio con df_codigos_limpio usando codigo_iso como clave
df = pd.merge(
    df_anio,
    df_codigos_limpio,
    on="codigo_iso",
    how="inner"
)

# Ordenar columnas principales si existen
columnas_orden = ["codigo_iso", "pais", "anio", "indice", "ranking"]
columnas_existentes = [col for col in columnas_orden if col in df.columns]
otras_columnas = [col for col in df.columns if col not in columnas_existentes]
df = df[columnas_existentes + otras_columnas]

print("\nDimensiones del DataFrame consolidado:")
print(df.shape)

df.head()

Columnas df_anio:
['codigo_iso', 'anio', 'indice', 'ranking']

Columnas df_codigos:
['codigo_iso', 'pais']

Códigos ISO asociados a más de un país:


,codigo_iso,cantidad_paises
179,ZWE,2



Registros conflictivos:


,codigo_iso,pais
179,ZWE,Zimbabue
180,ZWE,malo



Cantidad de registros en df_codigos original: 181
Cantidad de registros en df_codigos limpio: 180

Dimensiones del DataFrame consolidado:
(3060, 5)


,codigo_iso,pais,anio,indice,ranking
0,AFG,Afghanistán,2001,35.5,59.0
1,AGO,Angola,2001,30.2,50.0
2,ALB,Albania,2001,NaN,NaN
3,AND,Andorra,2001,NaN,NaN
4,ARE,Emiratos Árabes Unidos,2001,NaN,NaN


En esta etapa se consolidaron correctamente los archivos anuales del Índice de Libertad de Prensa. Primero se unieron las bases `libertad_prensa_01.csv` y `libertad_prensa_02.csv` mediante `pd.concat()`, normalizando previamente los nombres de columnas a minúsculas. El DataFrame anual resultante quedó con las columnas `codigo_iso`, `anio`, `indice` y `ranking`.

Luego se revisó la base de códigos de países y se detectó una inconsistencia en el código ISO `ZWE`, el cual aparecía asociado a dos nombres distintos: `Zimbabue` y `malo`. Para corregirlo, se eliminó el registro inconsistente y se conservó una sola relación válida entre código ISO y país.

Finalmente, se realizó la combinación entre la base anual y la base de códigos usando `codigo_iso` como clave. El DataFrame consolidado quedó con **3060 observaciones y 5 columnas**, incluyendo `codigo_iso`, `pais`, `anio`, `indice` y `ranking`. Esto indica que la unión fue realizada correctamente y que la base quedó preparada para el análisis posterior.



### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [3]:
# 2. Exploración inicial del conjunto de datos

print("Dimensiones del DataFrame:")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

print("\nNombres de columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
display(df.dtypes)

print("\nInformación general:")
df.info()

print("\nResumen estadístico:")
display(df.describe(include="all"))

# Estadísticos específicos de indice
print("\nEstadísticos de la columna indice:")
print("Mínimo:", df["indice"].min())
print("Máximo:", df["indice"].max())
print("Promedio:", round(df["indice"].mean(), 4))

# Países con valores extremos en indice
indice_min = df.loc[df["indice"].idxmin()]
indice_max = df.loc[df["indice"].idxmax()]

print("\nPaís con menor índice, mayor libertad de prensa:")
display(indice_min[["pais", "codigo_iso", "anio", "indice", "ranking"]])

print("\nPaís con mayor índice, menor libertad de prensa:")
display(indice_max[["pais", "codigo_iso", "anio", "indice", "ranking"]])

# Países con valores extremos en ranking
ranking_min = df.loc[df["ranking"].idxmin()]
ranking_max = df.loc[df["ranking"].idxmax()]

print("\nPaís con mejor ranking:")
display(ranking_min[["pais", "codigo_iso", "anio", "indice", "ranking"]])

print("\nPaís con peor ranking:")
display(ranking_max[["pais", "codigo_iso", "anio", "indice", "ranking"]])

# Valores nulos
nulos = df.isnull().sum()
porcentaje_nulos = (df.isnull().mean() * 100).round(2)

resumen_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje_nulos": porcentaje_nulos
})

print("\nValores nulos por columna:")
display(resumen_nulos)

print("\nColumnas con más de 30% de datos faltantes:")
display(resumen_nulos[resumen_nulos["porcentaje_nulos"] > 30])

# Unicidad y duplicados
print("\nCantidad de países distintos:")
print(df["pais"].nunique())

print("\nCantidad de años distintos:")
print(df["anio"].nunique())

print("\nAños disponibles:")
print(sorted(df["anio"].dropna().unique()))

print("\nFilas duplicadas exactas:")
print(df.duplicated().sum())

# Validación cruzada de codigo_iso y pais
validacion_iso = (
    df.groupby("codigo_iso")["pais"]
    .nunique()
    .reset_index(name="cantidad_paises")
    .query("cantidad_paises > 1")
)

print("\nCódigos ISO asociados a más de un país en df final:")
display(validacion_iso)

# También revisar si un mismo país aparece con más de un código
validacion_pais = (
    df.groupby("pais")["codigo_iso"]
    .nunique()
    .reset_index(name="cantidad_codigos")
    .query("cantidad_codigos > 1")
)

print("\nPaíses asociados a más de un código ISO en df final:")
display(validacion_pais)

Dimensiones del DataFrame:
Filas: 3060
Columnas: 5

Nombres de columnas:
['codigo_iso', 'pais', 'anio', 'indice', 'ranking']

Tipos de datos:


,0
codigo_iso,object
pais,object
anio,int64
indice,float64
ranking,float64



Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3060 non-null   object 
 1   pais        3060 non-null   object 
 2   anio        3060 non-null   int64  
 3   indice      2664 non-null   float64
 4   ranking     2837 non-null   float64
dtypes: float64(2), int64(1), object(2)
memory usage: 119.7+ KB

Resumen estadístico:


,codigo_iso,pais,anio,indice,ranking
count,3060,3060,3060.000000,2664.000000,2837.000000
unique,180,179,NaN,NaN,NaN
top,AFG,Nigeria,NaN,NaN,NaN
freq,17,34,NaN,NaN,NaN
mean,NaN,NaN,2009.941176,205.782316,477.930913
std,NaN,NaN,5.786024,2695.525264,6474.935347
min,NaN,NaN,2001.000000,0.000000,1.000000
25%,NaN,NaN,2005.000000,15.295000,34.000000
50%,NaN,NaN,2009.000000,28.000000,70.000000
75%,NaN,NaN,2015.000000,41.227500,110.000000



Estadísticos de la columna indice:
Mínimo: 0.0
Máximo: 64536.0
Promedio: 205.7823

País con menor índice, mayor libertad de prensa:


...

,nulos,porcentaje_nulos



Cantidad de países distintos:
179

Cantidad de años distintos:
17

Años disponibles:
[np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]

Filas duplicadas exactas:
0

Códigos ISO asociados a más de un país en df final:


,codigo_iso,cantidad_paises



Países asociados a más de un código ISO en df final:


,pais,cantidad_codigos
122,Nigeria,2


El DataFrame consolidado contiene **3060 filas y 5 columnas**. Las variables disponibles son `codigo_iso`, `pais`, `anio`, `indice` y `ranking`. Los tipos de datos son coherentes con el análisis: `codigo_iso` y `pais` son variables de texto, `anio` es entero, mientras que `indice` y `ranking` son numéricas.

La base contiene **180 códigos ISO**, **179 países distintos** y **17 años de observación**, entre 2001 y 2019. No existen filas duplicadas exactas, lo que indica que no hay registros repetidos completos. Sin embargo, aparece una inconsistencia menor: **Nigeria está asociada a más de un código ISO**, por lo que conviene mencionarlo como una limitación o detalle de calidad de datos.

Respecto a los valores faltantes, `indice` presenta **396 nulos**, equivalentes al **12,94%**, y `ranking` presenta **223 nulos**, equivalentes al **7,29%**. Ninguna columna supera el 30% de datos faltantes, por lo que la base puede seguir utilizándose sin eliminar columnas completas.

En los valores extremos, el menor índice corresponde a **Dinamarca en 2008**, con un índice de **0,0**, lo que representa una de las mejores situaciones de libertad de prensa dentro de la base. En cambio, el mayor índice corresponde a **Kosovo en 2014**, con **64536,0**, lo que aparece como un valor extremadamente alto. También Kosovo presenta el peor ranking en 2015, con **121056,0**. Estos valores son muy superiores al rango habitual del índice, por lo que podrían representar datos atípicos, problemas de escala o errores de registro que afectan los promedios y la variabilidad.




### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [5]:
# 3. Comparación regional: países de América considerados en la lista

america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

# Filtrar países de América presentes en la lista
df_america = df.loc[df["codigo_iso"].isin(america)].copy()

# Eliminar observaciones sin índice para evitar problemas con idxmin o idxmax
df_america = df_america.dropna(subset=["indice"])

print("Dimensiones del DataFrame filtrado para América:")
print(df_america.shape)

print("\nPaíses incluidos en el análisis:")
display(
    df_america[["codigo_iso", "pais"]]
    .drop_duplicates()
    .sort_values("pais")
)

# a) Solución con ciclo for
resultados_for = []

for anio in sorted(df_america["anio"].dropna().unique()):
    datos_anio = df_america.loc[df_america["anio"] == anio]

    # Como menor índice implica mayor libertad de prensa:
    mayor_libertad = datos_anio.loc[datos_anio["indice"].idxmin()]

    # Como mayor índice implica menor libertad de prensa:
    menor_libertad = datos_anio.loc[datos_anio["indice"].idxmax()]

    resultados_for.append({
        "anio": anio,
        "pais_mayor_libertad": mayor_libertad["pais"],
        "indice_minimo": mayor_libertad["indice"],
        "ranking_mayor_libertad": mayor_libertad["ranking"],
        "pais_menor_libertad": menor_libertad["pais"],
        "indice_maximo": menor_libertad["indice"],
        "ranking_menor_libertad": menor_libertad["ranking"]
    })

resultados_for = pd.DataFrame(resultados_for)

print("\nResultados usando ciclo for:")
display(resultados_for)

# b) Solución vectorizada con groupby
idx_min = df_america.groupby("anio")["indice"].idxmin()
idx_max = df_america.groupby("anio")["indice"].idxmax()

mayor_libertad_groupby = (
    df_america.loc[idx_min, ["anio", "pais", "indice", "ranking"]]
    .rename(columns={
        "pais": "pais_mayor_libertad",
        "indice": "indice_minimo",
        "ranking": "ranking_mayor_libertad"
    })
    .reset_index(drop=True)
)

menor_libertad_groupby = (
    df_america.loc[idx_max, ["anio", "pais", "indice", "ranking"]]
    .rename(columns={
        "pais": "pais_menor_libertad",
        "indice": "indice_maximo",
        "ranking": "ranking_menor_libertad"
    })
    .reset_index(drop=True)
)

resultados_groupby = pd.merge(
    mayor_libertad_groupby,
    menor_libertad_groupby,
    on="anio"
)

print("\nResultados usando groupby:")
display(resultados_groupby)

# Verificación: ambas soluciones deberían entregar los mismos países por año
comparacion = resultados_for[
    ["anio", "pais_mayor_libertad", "pais_menor_libertad"]
].merge(
    resultados_groupby[["anio", "pais_mayor_libertad", "pais_menor_libertad"]],
    on="anio",
    suffixes=("_for", "_groupby")
)

print("\nComparación entre ambas soluciones:")
display(comparacion)

Dimensiones del DataFrame filtrado para América:
(407, 5)

Países incluidos en el análisis:


,codigo_iso,pais
1807,ATG,Antigua y Barbuda
5,ARG,Argentina
1820,BLZ,Belize
21,BOL,Bolivia
22,BRA,Brasil
27,CAN,Canadá
29,CHL,Chile
35,COL,Colombia
38,CRI,Costa Rica
39,CUB,Cuba



Resultados usando ciclo for:


,anio,pais_mayor_libertad,indice_minimo,ranking_mayor_libertad,pais_menor_libertad,indice_maximo,ranking_menor_libertad
0,2001,Canadá,0.80,2.0,Cuba,90.30,99.0
1,2002,Trinidad y Tobago,1.00,2.0,Cuba,97.83,125.0
2,2003,Trinidad y Tobago,2.00,30.0,Argentina,35826.00,35.0
3,2004,Trinidad y Tobago,2.00,31.0,Cuba,87.00,112.0
4,2005,Bolivia,4.50,63.0,Cuba,95.00,109.0
5,2006,Canadá,4.88,84.0,Cuba,96.17,139.0
6,2007,Canadá,3.33,50.0,Cuba,88.33,117.0
7,2008,Canadá,3.70,62.0,Cuba,94.00,131.0
8,2009,Estados Unidos,6.75,115.0,Cuba,78.00,129.0
9,2012,Jamaica,9.88,176.0,Cuba,71.64,162.0



Resultados usando groupby:


,anio,pais_mayor_libertad,indice_minimo,ranking_mayor_libertad,pais_menor_libertad,indice_maximo,ranking_menor_libertad
0,2001,Canadá,0.80,2.0,Cuba,90.30,99.0
1,2002,Trinidad y Tobago,1.00,2.0,Cuba,97.83,125.0
2,2003,Trinidad y Tobago,2.00,30.0,Argentina,35826.00,35.0
3,2004,Trinidad y Tobago,2.00,31.0,Cuba,87.00,112.0
4,2005,Bolivia,4.50,63.0,Cuba,95.00,109.0
5,2006,Canadá,4.88,84.0,Cuba,96.17,139.0
6,2007,Canadá,3.33,50.0,Cuba,88.33,117.0
7,2008,Canadá,3.70,62.0,Cuba,94.00,131.0
8,2009,Estados Unidos,6.75,115.0,Cuba,78.00,129.0
9,2012,Jamaica,9.88,176.0,Cuba,71.64,162.0



Comparación entre ambas soluciones:


,anio,pais_mayor_libertad_for,pais_menor_libertad_for,pais_mayor_libertad_groupby,pais_menor_libertad_groupby
0,2001,Canadá,Cuba,Canadá,Cuba
1,2002,Trinidad y Tobago,Cuba,Trinidad y Tobago,Cuba
2,2003,Trinidad y Tobago,Argentina,Trinidad y Tobago,Argentina
3,2004,Trinidad y Tobago,Cuba,Trinidad y Tobago,Cuba
4,2005,Bolivia,Cuba,Bolivia,Cuba
5,2006,Canadá,Cuba,Canadá,Cuba
6,2007,Canadá,Cuba,Canadá,Cuba
7,2008,Canadá,Cuba,Canadá,Cuba
8,2009,Estados Unidos,Cuba,Estados Unidos,Cuba
9,2012,Jamaica,Cuba,Jamaica,Cuba


En este punto se filtraron los países de América incluidos en la lista entregada, obteniendo **407 observaciones y 5 columnas**. La lista incluye países latinoamericanos y también otros países del continente, como Canadá y Estados Unidos, por lo que el análisis se interpreta como una comparación regional de América.

Los resultados muestran que los países con mayor libertad de prensa relativa, es decir, aquellos con menor valor de `indice`, varían según el año. Entre los países que aparecen con mejores resultados se encuentran **Canadá**, **Trinidad y Tobago**, **Bolivia**, **Estados Unidos**, **Jamaica** y **Costa Rica**. Por ejemplo, Canadá aparece como el país con menor índice en 2001, 2006, 2007, 2008 y 2014; Jamaica destaca en 2012, 2013, 2018 y 2019; y Costa Rica aparece en 2015 y 2017.

Por otro lado, el país con menor libertad de prensa relativa es principalmente **Cuba**, que aparece con el mayor índice en casi todos los años analizados. Esto indica que, dentro del grupo de países considerados, Cuba presenta de forma persistente los peores resultados relativos en libertad de prensa. La única excepción relevante es **Argentina en 2003**, con un índice de **35826,0**, valor extremadamente alto que probablemente corresponde a un dato atípico o problema de escala, ya que se aleja demasiado del resto de observaciones regionales.

Finalmente, la solución con ciclo `for` y la solución con `groupby` entregan los mismos resultados. Esto valida que ambos métodos fueron implementados correctamente: el ciclo permite entender el procedimiento paso a paso, mientras que `groupby` entrega una solución más eficiente y compacta.

### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [6]:
# 4. Análisis anual del índice por país

# Tabla dinámica: filas = países, columnas = años, valores = índice máximo
tabla_pivot = pd.pivot_table(
    df,
    index="pais",
    columns="anio",
    values="indice",
    aggfunc="max",
    fill_value=0
)

print("Tabla dinámica del índice máximo por país y año:")
display(tabla_pivot.head())

# a) País con mayor valor de índice en toda la tabla y menor distinto de cero
max_valor = tabla_pivot.max().max()
pais_max = tabla_pivot.stack().idxmax()

tabla_sin_ceros = tabla_pivot.replace(0, np.nan)
min_valor = tabla_sin_ceros.min().min()
pais_min = tabla_sin_ceros.stack().idxmin()

print("\nMayor valor de índice en la tabla:")
print("País y año:", pais_max)
print("Índice:", max_valor)

print("\nMenor valor de índice distinto de cero en la tabla:")
print("País y año:", pais_min)
print("Índice:", min_valor)

# b) Años con promedio más alto y más bajo
promedio_anual = tabla_sin_ceros.mean(axis=0).sort_values(ascending=False)

print("\nPromedio anual del índice ordenado de mayor a menor:")
display(promedio_anual)

print("\nAño con promedio más alto:")
print(promedio_anual.idxmax(), round(promedio_anual.max(), 4))

print("\nAño con promedio más bajo:")
print(promedio_anual.idxmin(), round(promedio_anual.min(), 4))

# c) País con mayor variabilidad
variabilidad = tabla_sin_ceros.max(axis=1) - tabla_sin_ceros.min(axis=1)
variabilidad = variabilidad.sort_values(ascending=False)

print("\nPaíses con mayor variabilidad del índice:")
display(variabilidad.head(10))

print("\nPaís con mayor variabilidad:")
print(variabilidad.idxmax(), round(variabilidad.max(), 4))

# d) Países con índice constante a lo largo de todos los años registrados
# Se consideran solo valores distintos de cero para no confundir ausencia de datos con valor real.
constantes = tabla_sin_ceros.apply(lambda fila: fila.dropna().nunique() == 1, axis=1)
paises_constantes = tabla_sin_ceros.loc[constantes].index.tolist()

print("\nPaíses con índice constante en los años disponibles:")
print(paises_constantes)

# e) Países sin ningún dato
paises_sin_datos = tabla_pivot.loc[(tabla_pivot == 0).all(axis=1)].index.tolist()

print("\nPaíses sin ningún dato, todos los valores igual a 0:")
print(paises_sin_datos)

# Tabla resumen final con algunos indicadores por país
resumen_paises = pd.DataFrame({
    "indice_minimo": tabla_sin_ceros.min(axis=1),
    "indice_maximo": tabla_sin_ceros.max(axis=1),
    "indice_promedio": tabla_sin_ceros.mean(axis=1),
    "variabilidad": variabilidad
}).sort_values("variabilidad", ascending=False)

print("\nResumen por país:")
display(resumen_paises.head(10))

Tabla dinámica del índice máximo por país y año:


anio,2001,2002,2003,2004,2005,2006,2007,2008,2009,2012,2013,2014,2015,2017,2018,2019
pais,,,,,,,,,,,,,,,,
Afghanistán,35.5,40.17,28.25,39.17,44.25,56.50,59.25,54.25,51.67,37.36,37.07,37.44,37.75,39.46,37.28,36.55
Albania,0.0,6.50,11.50,14.17,18.00,25.50,16.00,21.75,21.50,30.88,29.92,28.77,29.92,29.92,29.49,29.84
Alemania,1.5,1.33,2.00,4.00,5.50,5.75,4.50,3.50,4.25,10.24,10.23,11.47,14.80,14.97,14.39,14.60
Algeria,31.0,33.00,43.50,40.33,40.00,40.50,31.33,49.56,47.33,36.54,36.26,36.63,41.69,42.83,43.13,45.75
Andorra,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,6.82,6.82,19.87,19.87,21.03,22.21,24.63



Mayor valor de índice en la tabla:
País y año: ('Kosovo', np.int64(2014))
Índice: 64536.0

Menor valor de índice distinto de cero en la tabla:
País y año: ('Austria', np.int64(2009))
Índice: 0.5

Promedio anual del índice ordenado de mayor a menor:


,0
anio,
2012,468.883509
2013,467.392384
2014,408.367093
2015,403.375029
2009,260.032733
2004,258.412075
2003,253.538050
2006,159.134720
2008,152.346358



Año con promedio más alto:
2012 468.8835

Año con promedio más bajo:
2002 25.8797

Países con mayor variabilidad del índice:


,0
pais,
Kosovo,46098.00
Tonga,37113.00
Senegal,37110.00
Argentina,35814.67
Bhutan,75.30
Sri Lanka,62.20
Myanmar,62.20
Irán,55.84
Maldivas,55.17



País con mayor variabilidad:
Kosovo 46098.0

Países con índice constante en los años disponibles:
['Antigua y Barbuda', 'Granada']

Países sin ningún dato, todos los valores igual a 0:
[]

Resumen por país:


,indice_minimo,indice_maximo,indice_promedio,variabilidad
pais,,,,
Kosovo,18438.00,64536.00,35657.800000,46098.00
Tonga,13.00,37126.00,2878.387692,37113.00
Senegal,14.00,37124.00,2341.349375,37110.00
Argentina,11.33,35826.00,2258.268125,35814.67
Bhutan,15.50,90.80,37.526875,75.30
Sri Lanka,15.80,78.00,49.400625,62.20
Myanmar,41.43,103.63,73.025000,62.20
Irán,48.30,104.14,77.892500,55.84
Maldivas,14.00,69.17,37.796667,55.17


La tabla dinámica permitió comparar la evolución del índice de libertad de prensa por país y año. En esta tabla, las filas corresponden a países, las columnas a años y los valores al índice máximo registrado. Los valores faltantes fueron reemplazados por 0, por lo que para ciertos cálculos fue necesario excluir esos ceros para no confundir ausencia de datos con valores reales del índice.

El mayor valor de índice en toda la tabla corresponde a **Kosovo en 2014**, con un valor de **64536,0**. Dado que un mayor índice representa una peor situación relativa de libertad de prensa, este resultado ubica a Kosovo como el caso más extremo del período analizado. Sin embargo, al igual que en el punto anterior, este valor parece ser un dato atípico o un problema de escala, porque se aleja considerablemente del rango habitual de la variable.

El menor valor distinto de cero corresponde a **Austria en 2009**, con un índice de **0,5**, lo que representa una de las mejores situaciones de libertad de prensa registradas en la base. En cuanto a los promedios anuales, el año con promedio más alto fue **2012**, con **468,88**, mientras que el promedio más bajo fue **2002**, con **25,88**. No obstante, los años con promedios más altos pueden estar influenciados por valores atípicos muy elevados, especialmente los casos de Kosovo, Tonga, Senegal o Argentina.

La variabilidad más alta también corresponde a **Kosovo**, con una diferencia de **46098,0** entre su máximo y mínimo. Luego aparecen Tonga, Senegal y Argentina, todos con variaciones muy elevadas. Esto sugiere que existen registros extremos que afectan el análisis de dispersión. Finalmente, los países con índice constante en los años disponibles son **Antigua y Barbuda** y **Granada**, mientras que no se detectaron países sin datos en todos los años.

### Conclusión general

El laboratorio permitió aplicar herramientas fundamentales de `pandas` para consolidar, limpiar, combinar y analizar datos del Índice de Libertad de Prensa. Se utilizaron funciones como `pd.concat()` para unir archivos anuales, `pd.merge()` para combinar información por código ISO, `groupby()` para obtener extremos por año y `pivot_table()` para construir una tabla comparativa por país y período.

Los resultados muestran que el índice permite comparar la libertad de prensa entre países y años, considerando que valores más bajos representan mejores condiciones. A nivel regional, países como Canadá, Jamaica, Costa Rica y Trinidad y Tobago aparecen en distintos años con mejores resultados relativos, mientras que Cuba se mantiene de forma persistente como el país con menor libertad de prensa dentro del grupo de América analizado.

Sin embargo, también se detectaron valores atípicos muy elevados, especialmente en Kosovo, Argentina, Tonga y Senegal. Estos valores influyen fuertemente en los promedios, máximos y medidas de variabilidad. Por ello, una conclusión importante del laboratorio es que, antes de interpretar los resultados de manera definitiva, es necesario revisar la calidad y escala de algunos registros extremos.

En general, la base consolidada quedó correctamente estructurada y permite realizar análisis comparativos, aunque se recomienda considerar los valores atípicos y algunas inconsistencias menores, como el caso de Nigeria asociado a más de un código ISO, antes de usar los resultados para conclusiones más profundas.